In [ ]:
!pip install python-docx textstat transformers torch bert-score sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

RIGOCHAR DETERMINISTA

In [ ]:
import os
import torch
import pyphen
import re
import numpy as np
import pandas as pd
from google.colab import userdata
from huggingface_hub import login
from google.colab import userdata
from bert_score import score
from docx import Document
from transformers import RobertaTokenizer, RobertaModel
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict

dic = pyphen.Pyphen(lang='en')
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

# --- DOCUMENTOS ---
"""
---- COMO SE TIENE ----
original = texto original
adaptaciones = generado por un modelo de lenguaje (_RE) adaptaciones = se refiere a (adaptado)
adaptado = generado por un humano experto (_LC) adaptado = se refiere a (reference)

---- COMO DEBE DE SER ----
original = texto original
reference = generado por un humano experto
adaptado = generado por un modelo de lenguaje
"""

def read_docx_text(filepath):
    doc = Document(filepath)
    return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])

def match_documents(doc_folder):
    files = [f for f in os.listdir(doc_folder) if f.endswith(".docx")]
    paired = {}

    for f in files:
      filename = f.replace(".docx", "")
      full_path = os.path.join(doc_folder,f)

      # Casos clínicos - adaptados por humanos (reference)
      if filename.endswith("_LC"):
         base = filename.replace("_LC", "")
         paired.setdefault(base, {})["reference"] = full_path
         paired[base]["grupo"] = "notas_clinicas"

      # Casos clínicos - adaptados por modelo (adaptado)
      elif filename.endswith("_AD"):
      #elif filename.endswith("_CRE"):
         base = filename.replace("_AD", "")
         #base = filename.replace("_CRE", "")
         paired.setdefault(base, {})["adapted"] = full_path
         paired[base]["grupo"] = "notas_clinicas"

      #CM-AAPP - adaptados por humanos (reference)
      elif filename.endswith(" - LC"):
         base = filename.replace(" - LC", "")
         paired.setdefault(base, {})["reference"] = full_path
         paired[base]["grupo"] = "cm_aapp"

      #CM-AAPP - adaptados por modelo (adaptado)
      elif filename.endswith(" - AD"):
      #elif filename.endswith(" - CRE"):
         base = filename.replace(" - AD", "")
         #base = filename.replace(" - CRE", "")
         paired.setdefault(base, {})["adapted"] = full_path
         paired[base]["grupo"] = "cm_aapp"

      # Texto original
      else:
        base = filename
        paired.setdefault(base, {})["original"] = full_path
        #paired[base]["grupo"] = "original"

    final_pairs = {
        k: v for k, v in paired.items()
        if "original" in v and "reference" in v and "adapted" in v
    }

    print(f" Archivos detectados: {len(files)}")
    print(f"✅ Pares válidos encontrados: {len(final_pairs)}")
    return final_pairs

# ---- NUMERO DE PALABRAS ----
def count_words(text):
    """
    Cuenta el número de palabras del texto
    """
    return len(text.split())

def contar_silabas(palabra):
    palabra = palabra.lower()
    replacements = {'á':'a', 'é':'e', 'í':'i', 'ó':'o', 'ú':'u', 'ü':'u'}
    for k, v in replacements.items():
        palabra = palabra.replace(k, v)
    return max(dic.inserted(palabra).count('-') + 1, 1)

# --- FUNCIONES DE MÉTRICAS ---
"""
original = texto original
reference = referencia humana
adapted = adaptacionmes del modelo
"""

# ----- Semantic Preservation -----
#BERTScore
def calculate_bertscore(original, reference):
    """
    Calcula el BERTScore
    """
    P, R, F1 = score([reference], [original], lang="es", rescale_with_baseline=True)
    return P.mean().item(), R.mean().item(), F1.mean().item()

#Semantic_Answer_Similarity (SAS RoBERTA)
def calculate_sas(original, reference, model, tokenizer):
    inputs_orig = tokenizer(original, return_tensors="pt", truncation=True, padding=True)
    inputs_simp = tokenizer(reference, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        emb_orig = model(**inputs_orig).last_hidden_state.mean(dim=1)
        emb_simp = model(**inputs_simp).last_hidden_state.mean(dim=1)

    similarity = torch.nn.functional.cosine_similarity(emb_orig, emb_simp).item()
    return similarity

#Semantic_Answer_Similarity (SAS sentence-transformers)
def SAS(original, reference, st_model):
    emb1 = st_model.encode([original], convert_to_tensor=True)
    emb2 = st_model.encode([reference], convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()

# ----- Readability for Spanish-----
#Fernandez Huerta
def calculate_fh(text):
    frases = re.split(r'(?<=[.!?])\s+', text)
    #frases = re.split(r'(?<=[.!?])[\s\n]+', text)
    frases = [f.strip() for f in frases if f.strip()]
    if not frases:
        return None

    palabras = [p for f in frases for p in re.findall(r'\b\w+\b', f)]
    total_silabas = sum(contar_silabas(p) for p in palabras)

    if len(palabras) == 0:
        return None

    silabas_por_palabra = total_silabas / len(palabras)
    frases_por_palabra = len(frases) / len(palabras)

    indice = 206.84 - (60 * silabas_por_palabra) - (102 * frases_por_palabra)
    return indice

# ----- Specific Simplification -----
#SARI
def calculate_sari(source, prediction, references):
    """
    Calcula el SARI (ADD, KEEP, DELETE) siguiendo el paper Optimizing Statistical Machine Translation for Text Simplification de Xu et al. (2016) y
    devuelve también la media de SARI.
    """
    def ngram_counter(sentence, n):
        """
         n-gramas en una oración.
        """
        return [tuple(sentence[i:i+n]) for i in range(len(sentence)-n+1)] #set

    def precision_recall_f1(tp, fp, fn):
        """
        Calcula precisión, recall y F1-score.
        """
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
        return precision, recall, f1

    # Separa en palabras
    source_tokens = source.split()
    prediction_tokens = prediction.split()
    references_tokens = [ref.split() for ref in references]

    # Defini n-gramas para diferentes niveles (1-gram, 2-gram, etc.)
    ngram_levels = [1, 2, 3, 4]

    add_scores, keep_scores, delete_scores = [], [], []

    for n in ngram_levels:
        source_ngrams = set(ngram_counter(source_tokens, n))
        pred_ngrams = set(ngram_counter(prediction_tokens, n))
        ref_ngrams = [set(ngram_counter(ref, n)) for ref in references_tokens]

        # Une todas las referencias
        union_ref_ngrams = set().union(*ref_ngrams)

        keep_ngrams = source_ngrams.intersection(pred_ngrams)
        add_ngrams = pred_ngrams.difference(source_ngrams)
        del_ngrams = source_ngrams.difference(pred_ngrams)

        # --- KEEP ---
        tp_keep = len(keep_ngrams.intersection(union_ref_ngrams))
        fp_keep = len(keep_ngrams.difference(union_ref_ngrams))
        fn_keep = len(source_ngrams.intersection(union_ref_ngrams).difference(keep_ngrams))

        # --- ADD ---
        tp_add = len(add_ngrams.intersection(union_ref_ngrams))
        fp_add = len(add_ngrams.difference(union_ref_ngrams))
        fn_add = len(union_ref_ngrams.difference(source_ngrams).difference(add_ngrams))

        # --- DELETE ---
        good_deletions = source_ngrams.difference(union_ref_ngrams)
        tp_del = len(del_ngrams.intersection(good_deletions))
        fp_del = len(del_ngrams.difference(good_deletions))

        keep_precision, keep_recall, keep_f1 = precision_recall_f1(tp_keep, fp_keep, fn_keep)
        add_precision, add_recall, add_f1 = precision_recall_f1(tp_add, fp_add, fn_add)
        del_precision = tp_del / (tp_del + fp_del) if (tp_del + fp_del) > 0 else 0
        del_f1 = del_precision

        add_scores.append(add_f1)
        keep_scores.append(keep_f1)
        delete_scores.append(del_f1)

    # Promedios finales
    avg_add = sum(add_scores) / len(add_scores)
    avg_keep = sum(keep_scores) / len(keep_scores)
    avg_del = sum(delete_scores) / len(delete_scores)
    sari = (avg_add + avg_keep + avg_del) / 3

    return avg_add, avg_keep, avg_del, sari

# === EVALUACIÓN DE MÉTRICAS YA IMPLEMENTADAS ===
"""
# path_original = textos originales
# path_reference = textos referencia humana (LC)
# path_adapted = textos adaptados por un modelo
"""
def evaluate(name, path_original, path_adapted, path_reference, model_roberta, tokenizer, st_model):
    original = read_docx_text(path_original)
    reference = read_docx_text(path_reference)
    adapted = read_docx_text(path_adapted)
    resultado = {"documento": name}
    errores = []

    try:
        resultado["nº palabras (Original)"] = len(original.split())
        resultado["nº palabras (Referencia)"] = len(reference.split())
        resultado["nº palabras (Adaptado)"] = len(adapted.split())

        # FH
        fh_o = calculate_fh(original)
        resultado["Fernandez Huerta O"] = round(fh_o, 2)
        fh_r = calculate_fh(reference)
        resultado["Fernandez Huerta R"] = round(fh_r, 2)
        fh_a = calculate_fh(adapted)
        resultado["Fernandez Huerta A"] = round(fh_a, 2)

        # SARI
        add, keep, delete, sari = calculate_sari(original, adapted, [reference]) #original, adapted, [reference]
        resultado.update({
            "ADD": round(add, 4),
            "KEEP": round(keep, 4),
            "DELETE": round(delete, 4),
            "SARI_MEDIA": round(sari, 4),
        })

        # BERTScore
        P, R, F1 = calculate_bertscore(original, reference)
        resultado["BERTScore_F1"] = round(F1, 4)

        # SAS RoBERTa
        sas_roberta = calculate_sas(original, reference, model_roberta, tokenizer)
        resultado["SAS_RoBERTa"] = round(sas_roberta, 4)

        # SAS SentenceTransformer
        sas_st = SAS(original, reference, st_model)
        resultado["SAS_ST"] = round(sas_st, 4) #(adaptado, reference, stmodel) se necesita (adptado) y la referencia
        #referencia (adaptado hechos por los humanos el que ya tenemos)

    except Exception as e:
        print(f" Error en '{name}': {e}")
        errores.append(name)

    return resultado

# ---- MAIN ----
if __name__ == "__main__":
  doc_folder = "/content/Documentos"

  #SAS
  tokenizer_roberta = RobertaTokenizer.from_pretrained("roberta-base")
  model_roberta = RobertaModel.from_pretrained("roberta-base")
  st_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

  doc_pairs = match_documents(doc_folder)
  print(f"Pares encontrados: {len(doc_pairs)}")

  results_casos_clinicos = []
  results_cm_aapp = []
  errores = []

  for name, paths in doc_pairs.items():
    print(f"Evaluando: {name}")
    result = evaluate(name, paths['original'], paths['adapted'], paths['reference'], model_roberta, tokenizer_roberta, st_model)

    if paths.get("grupo") == "notas_clinicas":
      results_casos_clinicos.append(result)
    elif paths.get("grupo") == "cm_aapp":
      results_cm_aapp.append(result)

  output_path = "/content/Metrics_colab_DE.xlsx"
  with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
      if results_casos_clinicos:
          pd.DataFrame(results_casos_clinicos).to_excel(writer, sheet_name="Casos clinicos", index=False)
      if results_cm_aapp:
          pd.DataFrame(results_cm_aapp).to_excel(writer, sheet_name="CM-AAPP", index=False)

  print("✅ Excel generado con hojas separadas.")

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

... (15 outputs left out)

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Evaluando: CasoClinico2020-16-34
Evaluando: CasoClinico2020-3-6
Evaluando: IRSST
Evaluando: CasoClinico2020-21-43
Evaluando: CasoClinico2020-26-61
Evaluando: CEPA Sierra Guadarrama
Evaluando: CasoClinico2020-6-14
Evaluando: DG Atención al Mayor
Evaluando: CasoClinico2020-25-57
Evaluando: Junta Superior de Hacienda
Evaluando: IES Fernando VI
Evaluando: IES Benjamín Rúa
Evaluando: Residencias de mayores
Evaluando: Red oficinas turismo
Evaluando: IES Leonardo da Vinci
Evaluando: IES San Blas
Evaluando: Agencia AMAS
Evaluando: IES Arquitecto Pedro Gumiel
Evaluando: CasoClinico2020-14-32
Evaluando: CasoClinico2020-23-47
Evaluando: Residencia Navacerrada
Evaluando: CIFP Tomas y Valiente
Evaluando: CEPA Paulo Freire
Evaluando: CasoClinico2020-11-26
Evaluando: CasoClinico2020-22-44
Evaluando: IES Los Rosales
Evaluando: CasoClinico2020-5-8
Evaluando: DG Atención Discapacidad
Evaluando: CasoClinico2020-1-2
Evaluando: Instalación Canal Isabel II
Evaluando: CasoClinico2020-2-5
Evaluando: CasoClini

METRICAS CON RIGOCHAR CREATIVO

In [ ]:
import os
import torch
import pyphen
import re
import numpy as np
import pandas as pd
from google.colab import userdata
from huggingface_hub import login
from google.colab import userdata
from bert_score import score
from docx import Document
from transformers import RobertaTokenizer, RobertaModel
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict
#from alignscore import AlignScore
#from questeval.questeval_metric import QuestEval

dic = pyphen.Pyphen(lang='en')
hf_token = userdata.get('HF_TOKEN')
login(hf_token)

# --- DOCUMENTOS ---
"""
---- COMO SE TIENE ----
original = texto original
adaptaciones = generado por un modelo de lenguaje (_RE) adaptaciones = se refiere a (adaptado)
adaptado = generado por un humano experto (_LC) adaptado = se refiere a (reference)

---- COMO DEBE DE SER ----
original = texto original
reference = generado por un humano experto
adaptado = generado por un modelo de lenguaje
"""

def read_docx_text(filepath):
    doc = Document(filepath)
    return "\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])

def match_documents(doc_folder):
    files = [f for f in os.listdir(doc_folder) if f.endswith(".docx")]
    paired = {}

    for f in files:
      filename = f.replace(".docx", "")
      full_path = os.path.join(doc_folder,f)

      # Casos clínicos - adaptados por humanos (reference)
      if filename.endswith("_LC"):
         base = filename.replace("_LC", "")
         paired.setdefault(base, {})["reference"] = full_path
         paired[base]["grupo"] = "notas_clinicas"

      # Casos clínicos - adaptados por modelo (adaptado)
      #elif filename.endswith("_AD"):
      elif filename.endswith("_CRE"):
         #base = filename.replace("_AD", "")
         base = filename.replace("_CRE", "")
         paired.setdefault(base, {})["adapted"] = full_path
         paired[base]["grupo"] = "notas_clinicas"

      #CM-AAPP - adaptados por humanos (reference)
      elif filename.endswith(" - LC"):
         base = filename.replace(" - LC", "")
         paired.setdefault(base, {})["reference"] = full_path
         paired[base]["grupo"] = "cm_aapp"

      #CM-AAPP - adaptados por modelo (adaptado)
      #elif filename.endswith(" - AD"):
      elif filename.endswith(" - CRE"):
         #base = filename.replace(" - AD", "")
         base = filename.replace(" - CRE", "")
         paired.setdefault(base, {})["adapted"] = full_path
         paired[base]["grupo"] = "cm_aapp"

      # Texto original
      else:
        base = filename
        paired.setdefault(base, {})["original"] = full_path
        #paired[base]["grupo"] = "original"

    final_pairs = {
        k: v for k, v in paired.items()
        if "original" in v and "reference" in v and "adapted" in v
    }

    print(f" Archivos detectados: {len(files)}")
    print(f"✅ Pares válidos encontrados: {len(final_pairs)}")
    return final_pairs

# ---- NUMERO DE PALABRAS ----
def count_words(text):
    """
    Cuenta el número de palabras del texto
    """
    return len(text.split())

def contar_silabas(palabra):
    palabra = palabra.lower()
    replacements = {'á':'a', 'é':'e', 'í':'i', 'ó':'o', 'ú':'u', 'ü':'u'}
    for k, v in replacements.items():
        palabra = palabra.replace(k, v)
    return max(dic.inserted(palabra).count('-') + 1, 1)

# --- FUNCIONES DE MÉTRICAS ---
"""
original = texto original
reference = referencia humana
adapted = adaptacionmes del modelo
"""

#BERTScore
def calculate_bertscore(original, reference):
    """
    Calcula el BERTScore
    """
    P, R, F1 = score([reference], [original], lang="es", rescale_with_baseline=True)
    return P.mean().item(), R.mean().item(), F1.mean().item()

#Fernandez Huerta
def calculate_fh(text):
    frases = re.split(r'(?<=[.!?])\s+', text)
    #frases = re.split(r'(?<=[.!?])[\s\n]+', text)
    frases = [f.strip() for f in frases if f.strip()]
    if not frases:
        return None

    palabras = [p for f in frases for p in re.findall(r'\b\w+\b', f)]
    total_silabas = sum(contar_silabas(p) for p in palabras)

    if len(palabras) == 0:
        return None

    silabas_por_palabra = total_silabas / len(palabras)
    frases_por_palabra = len(frases) / len(palabras)

    indice = 206.84 - (60 * silabas_por_palabra) - (102 * frases_por_palabra)
    return indice

#SARI
def calculate_sari(source, prediction, references):
    """
    Calcula el SARI (ADD, KEEP, DELETE) siguiendo el paper Optimizing Statistical Machine Translation for Text Simplification de Xu et al. (2016) y
    devuelve también la media de SARI.
    """
    def ngram_counter(sentence, n):
        """
         n-gramas en una oración.
        """
        return [tuple(sentence[i:i+n]) for i in range(len(sentence)-n+1)] #set

    def precision_recall_f1(tp, fp, fn):
        """
        Calcula precisión, recall y F1-score.
        """
        precision = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
        return precision, recall, f1

    # Separa en palabras
    source_tokens = source.split()
    prediction_tokens = prediction.split()
    references_tokens = [ref.split() for ref in references]

    # Defini n-gramas para diferentes niveles (1-gram, 2-gram, etc.)
    ngram_levels = [1, 2, 3, 4]

    add_scores, keep_scores, delete_scores = [], [], []

    for n in ngram_levels:
        source_ngrams = set(ngram_counter(source_tokens, n))
        pred_ngrams = set(ngram_counter(prediction_tokens, n))
        ref_ngrams = [set(ngram_counter(ref, n)) for ref in references_tokens]

        # Une todas las referencias
        union_ref_ngrams = set().union(*ref_ngrams)

        keep_ngrams = source_ngrams.intersection(pred_ngrams)
        add_ngrams = pred_ngrams.difference(source_ngrams)
        del_ngrams = source_ngrams.difference(pred_ngrams)

        # --- KEEP ---
        tp_keep = len(keep_ngrams.intersection(union_ref_ngrams))
        fp_keep = len(keep_ngrams.difference(union_ref_ngrams))
        fn_keep = len(source_ngrams.intersection(union_ref_ngrams).difference(keep_ngrams))

        # --- ADD ---
        tp_add = len(add_ngrams.intersection(union_ref_ngrams))
        fp_add = len(add_ngrams.difference(union_ref_ngrams))
        fn_add = len(union_ref_ngrams.difference(source_ngrams).difference(add_ngrams))

        # --- DELETE ---
        good_deletions = source_ngrams.difference(union_ref_ngrams)
        tp_del = len(del_ngrams.intersection(good_deletions))
        fp_del = len(del_ngrams.difference(good_deletions))

        keep_precision, keep_recall, keep_f1 = precision_recall_f1(tp_keep, fp_keep, fn_keep)
        add_precision, add_recall, add_f1 = precision_recall_f1(tp_add, fp_add, fn_add)
        del_precision = tp_del / (tp_del + fp_del) if (tp_del + fp_del) > 0 else 0
        del_f1 = del_precision

        add_scores.append(add_f1)
        keep_scores.append(keep_f1)
        delete_scores.append(del_f1)

    # Promedios finales
    avg_add = sum(add_scores) / len(add_scores)
    avg_keep = sum(keep_scores) / len(keep_scores)
    avg_del = sum(delete_scores) / len(delete_scores)
    sari = (avg_add + avg_keep + avg_del) / 3

    return avg_add, avg_keep, avg_del, sari

#Semantic_Answer_Similarity (SAS RoBERTA)
def calculate_sas(original, reference, model, tokenizer):
    inputs_orig = tokenizer(original, return_tensors="pt", truncation=True, padding=True)
    inputs_simp = tokenizer(reference, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        emb_orig = model(**inputs_orig).last_hidden_state.mean(dim=1)
        emb_simp = model(**inputs_simp).last_hidden_state.mean(dim=1)

    similarity = torch.nn.functional.cosine_similarity(emb_orig, emb_simp).item()
    return similarity

#Semantic_Answer_Similarity (SAS sentence-transformers)
def SAS(original, reference, st_model):
    emb1 = st_model.encode([original], convert_to_tensor=True)
    emb2 = st_model.encode([reference], convert_to_tensor=True)
    return util.cos_sim(emb1, emb2).item()


# === EVALUACIÓN DE MÉTRICAS YA IMPLEMENTADAS ===
"""
# path_original = textos originales
# path_reference = textos referencia humana (LC)
# path_adapted = textos adaptados por un modelo
"""
def evaluate(name, path_original, path_adapted, path_reference, model_roberta, tokenizer, st_model):
    original = read_docx_text(path_original)
    reference = read_docx_text(path_reference)
    adapted = read_docx_text(path_adapted)
    resultado = {"documento": name}
    errores = []

    try:
        resultado["nº palabras (Original)"] = len(original.split())
        resultado["nº palabras (Referencia)"] = len(reference.split())
        resultado["nº palabras (Adaptado)"] = len(adapted.split())

        # FH
        fh_o = calculate_fh(original)
        resultado["Fernandez Huerta O"] = round(fh_o, 2)
        fh_r = calculate_fh(reference)
        resultado["Fernandez Huerta R"] = round(fh_r, 2)
        fh_a = calculate_fh(adapted)
        resultado["Fernandez Huerta A"] = round(fh_a, 2)

        # SARI
        add, keep, delete, sari = calculate_sari(original, adapted, [reference]) #original, adapted, [reference]
        resultado.update({
            "ADD": round(add, 4),
            "KEEP": round(keep, 4),
            "DELETE": round(delete, 4),
            "SARI_MEDIA": round(sari, 4),
        })

        # BERTScore
        P, R, F1 = calculate_bertscore(original, reference)
        resultado["BERTScore_F1"] = round(F1, 4)

        # SAS RoBERTa
        sas_roberta = calculate_sas(original, reference, model_roberta, tokenizer)
        resultado["SAS_RoBERTa"] = round(sas_roberta, 4)

        # SAS SentenceTransformer
        sas_st = SAS(original, reference, st_model)
        resultado["SAS_ST"] = round(sas_st, 4) #(adaptado, reference, stmodel) se necesita (adptado) y la referencia
        #referencia (adaptado hechos por los humanos el que ya tenemos)

    except Exception as e:
        print(f" Error en '{name}': {e}")
        errores.append(name)

    return resultado

# ---- MAIN ----
if __name__ == "__main__":
  doc_folder = "/content/Documentos1"

  #SAS
  tokenizer_roberta = RobertaTokenizer.from_pretrained("roberta-base")
  model_roberta = RobertaModel.from_pretrained("roberta-base")
  st_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')

  doc_pairs = match_documents(doc_folder)
  print(f"Pares encontrados: {len(doc_pairs)}")

  results_casos_clinicos = []
  results_cm_aapp = []
  errores = []

  for name, paths in doc_pairs.items():
    print(f"Evaluando: {name}")
    result = evaluate(name, paths['original'], paths['adapted'], paths['reference'], model_roberta, tokenizer_roberta, st_model)

    if paths.get("grupo") == "notas_clinicas":
      results_casos_clinicos.append(result)
    elif paths.get("grupo") == "cm_aapp":
      results_cm_aapp.append(result)

  output_path = "/content/Metrics_colab_CRE.xlsx"
  with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
      if results_casos_clinicos:
          pd.DataFrame(results_casos_clinicos).to_excel(writer, sheet_name="Casos clinicos", index=False)
      if results_cm_aapp:
          pd.DataFrame(results_cm_aapp).to_excel(writer, sheet_name="CM-AAPP", index=False)

  print("✅ Excel generado con hojas separadas.")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


 Archivos detectados: 150
✅ Pares válidos encontrados: 50
Pares encontrados: 50
Evaluando: CasoClinico2020-16-34
Evaluando: CasoClinico2020-3-6
Evaluando: CIFP Tomas y Valiente
Evaluando: CasoClinico2020-14-32
Evaluando: CEPA Sierra Guadarrama
Evaluando: CasoClinico2020-21-43
Evaluando: IES Los Rosales
Evaluando: CasoClinico2020-6-14
Evaluando: Junta Superior de Hacienda
Evaluando: Red oficinas turismo
Evaluando: IES Leonardo da Vinci
Evaluando: IRSST
Evaluando: IES San Blas
Evaluando: IES Fernando VI
Evaluando: CasoClinico2020-10-23
Evaluando: Agencia AMAS
Evaluando: IES Arquitecto Pedro Gumiel
Evaluando: Residencias de mayores
Evaluando: Residencia Navacerrada
Evaluando: CEPA Paulo Freire
Evaluando: CasoClinico2022-27-15
Evaluando: CasoClinico2020-22-44
Evaluando: CasoClinico2020-5-8
Evaluando: DG Atención Discapacidad
Evaluando: CasoClinico2020-11-26
Evaluando: CasoClinico2020-1-2
Evaluando: Instalación Canal Isabel II
Evaluando: CIFP José Luis Garci
Evaluando: IES Benjamín Rúa
Eval